<a href="https://colab.research.google.com/github/Oaimtac/farm-soccer/blob/main/%E5%BF%83%E9%9B%BB%E5%9C%96%E8%A8%8A%E8%99%9F%E7%96%B2%E5%8B%9E%E5%88%86%E6%9E%90%E5%8E%9F%E7%90%86%E8%88%87%E5%AF%A6%E4%BD%9C(II)_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **心電圖訊號疲勞分析原理與實作(II)_2**

# 0. 先收錄一些會用到的公式和函式吧！

In [ ]:
from scipy.fft import fft, fftfreq, ifft #從scipy.fft函式庫中，引入頻域轉換公式fft, fftfreq, ifft
import plotly.graph_objects as go     #引入plotly.graph_objects函式庫，命名為go
import numpy as np             #引入numpy函式庫，命名為np
import pandas as pd            #引入pandas函式庫，命名為np
from google.colab import files      #從google colab函式庫中，引入files函式

# 1.匯入錄製長度5分鐘以上的「範例心電訊號(6分鐘)」來學習接下來的課程吧

In [ ]:
#先將資料夾中的「我的心電訊號」檔案放到Google雲端處理器中
uploaded = files.upload()            #用uploaded來處理上傳檔案程序
for fn in uploaded.keys():           #將選取所有上傳檔案的名字印出
  print('你已經上傳了','"{name}" '.format(
      name=fn, length=len(uploaded[fn])))

# 2.將前一章所實作的波峰偵測部分集中在此區塊程式


In [ ]:
df = pd.read_csv('範例心電訊號(6分鐘).txt') #以pandas的read_csv即可讀取檔案
data = np.array(df)          #將讀取到的檔案換成我們習慣的numpy陣列來處理
data2 = data[:,0]           #指定data中的第一行資料，建立陣列data2

#強大的濾波器，不需要再轉換到頻域即可快速濾除雜訊，還能達到一樣的效果！
from scipy.signal import butter, filtfilt #從scipy.signal函式庫中，引入濾波器函式butter, filtfilt
def super_filter(data, frequency, save_frequency_start, save_frequency_end):   #super_filter(時域訊號, 週期, 特定保留頻率起點, 特定保留頻率終點)
  b, a = butter(3, [save_frequency_start, save_frequency_end], fs=frequency, btype='band')
  y = filtfilt(b, a, data)
  return y

data3 = data[:,0]           #指定data中的第一行資料，重新建立一筆陣列data3
period = 1/500     #宣告資料取樣週期參數
frequency = 500     #宣告資料取樣頻率參數

data4 = super_filter(data3, frequency, 15, 35) #將時域訊號, 資料取樣頻率, 心電訊號R波保留頻率起點, 心電訊號R波保留頻率終點代入強大的濾波器(super filter)


from scipy.signal import find_peaks #從scipy.signal函式庫中，引入找波峰函式find_peaks
height=0.1    #宣告波峰認定資料最低限值
distance=140  #宣告,波峰之間間隔資料點最低限值
peak_list_x = find_peaks(data4, height=height, distance=distance)[0]  #指定其x軸的位置為波峰偵測位置
peak_list_y = [data4[j] for j in peak_list_x]              #指定其y軸的位置為波峰偵測位置時的data5對應數值(R波特化)

#畫出濾出R段後的波形與波峰偵測位置
fig2 = go.Figure()                #建立一個圖形物件
fig2.add_trace(go.Scatter(             #新增一條線條在此圖形
    y=data4,                #新增一條線條，，將轉換訊號的時域部分畫出
    name='範例心電訊號(濾出R波段)'    #幫此線條命名
))
fig2.add_trace(go.Scatter(           #新增一條線條在此圖形
    x=peak_list_x,              #指定其x軸的位置
    y=peak_list_y,              #指定其y軸的位置
    mode='markers',             #定義此線段不連線，僅畫出有標記的位置
    marker=dict(
        color='red',           #幫此標記以紅色標記
    ),
    name='分析之波峰位置'          #幫此線條命名
))
fig2.update_layout(                  #更新圖形的說明
    title="範例心電訊號(濾出R波段)與偵測波峰", #幫這張圖形物件命名
    font=dict(                  #設定圖形名稱的文字
        family="Courier New, monospace",  #字體類型設定：Courier New, monospace
        size=20,               #字體大小設定：20
        color="RebeccaPurple"         #字體顏色設定：RebeccaPurple
    )
)
fig2.show()                 #顯示圖形

# 3.將前一章所實作的4Hz再取樣部分集中在此區塊程式


In [ ]:
peak_peak_list = np.diff(peak_list_x) #兩兩波峰相減計算波峰之間的距離

peak_list_x_last = peak_list_x[-1]           #一個陣列中的最後一個數值，可以用array[-1]取得
peak_list_length = peak_list_x[-1] - peak_list_x[0] #將心電訊號的最後一個波峰位置，減去第一個波峰位置，就能得知心電訊號在首尾波峰之間總共有幾個資料點，宣告為peak_list_length。

data4_seconds = peak_list_length/500              #將心電訊號首尾波峰之間資料總數除以500，得出訊號總秒數
new_4Hz_peaks_peaks_list_x_length =  int(data4_seconds*4)  #將總秒數乘以4，得知新建立的波峰間距陣列的資料總長需要設定多少

n = 0           #n設定為0，對陣列來說代表從第一個資料點開始
pointer = peak_list_x[n] #從第一個波峰位置開始推移
pointer_speed = 125    #從500Hz的原始資料紀錄頻率，再取樣為4Hz頻率的資料，原始資料每推移125即可執行一次4Hz的資料紀錄，紀錄的數值為波峰間距
pointer_counter = n+1   #推移的過程，需要與下一個波峰位置比較，若發現已經超過下一個波峰位置，就要將紀錄內容更換到正確的波峰間距數值

new_4Hz_peaks_peaks_list = np.zeros(new_4Hz_peaks_peaks_list_x_length) #建立一個空的4Hz波峰間距陣列
counter = 0                                 #定位空陣列的資料位置，幫一個空陣列資料填入數值後要記得加一

for i in range(new_4Hz_peaks_peaks_list_x_length):    #用迴圈把正確的波峰間距，填入預設空的4Hz波峰間距陣列
  if(pointer > peak_list_x[pointer_counter]):  #每次都要先與下一個波峰位置比較，若發現已經超過下一個波峰位置，就要將紀錄內容更換到對應的波峰間距數值
    pointer_counter = pointer_counter + 1  #若有發現超過，就要加一，來更換到正確的波峰間距數值
  pointer = pointer + pointer_speed       #以預設好的推移速度pointer_speed，將pointer向前推移

  new_4Hz_peaks_peaks_list[counter] = peak_peak_list[pointer_counter - 1]   #將正確的波峰間距填入空的4Hz波峰間距陣列；pointer_counter從1開始，所以這邊要減一，從波峰間距的起始位置(0)開始填入
  counter = counter + 1                              #定位空陣列的資料位置，幫一個空陣列資料填入數值後要記得加一

#畫出4Hz再取樣後的波峰間距陣列波形
fig = go.Figure()                #建立一個圖形物件
fig.add_trace(go.Scatter(             #新增一條線條在此圖形
    y=new_4Hz_peaks_peaks_list,        #新增一條線條，將轉換訊號的畫出
    mode='markers',               #定義此線段不連線，僅畫出有標記的位置
))
fig.update_layout(                  #更新圖形的說明
    title="範例心電訊號(6分鐘)(4Hz再取樣的波峰陣列)", #幫這張圖形物件命名
    font=dict(                  #設定圖形名稱的文字
        family="Courier New, monospace",  #字體類型設定：Courier New, monospace
        size=20,               #字體大小設定：20
        color="RebeccaPurple"         #字體顏色設定：RebeccaPurple
    )
)

fig.show()                 #顯示圖形

# 4.利用頻域轉換公式，將波峰間距陣列轉換到頻域

In [ ]:
data5 = new_4Hz_peaks_peaks_list        #宣告data5變數，代稱名字很長的「new_4Hz_peaks_peaks_list」
dots = len(data5)                #取得波峰間距陣列資料總數
period = 1/4                  #宣告週期參數

xf = fftfreq(dots, period)[1: int(dots/2)]   #fftfreq為頻域x軸的轉換公式，需要加入y值訊號所含的點數與週期，轉換後得出頻譜分布圖的x值陣列。
                           #頻域轉換公式只需取前二分之一有效值

yf = fft(data5)                  #fft 為頻域y軸的轉換公式，轉換後得出頻譜分布圖的y值陣列，
yf_half = np.abs(yf[1: int(dots/2)])        #頻域轉換公式只需取前二分之一有效值
yf_normalized = 2 / dots * yf_half        #頻域轉換後將資料正規畫

fig = go.Figure()                 #建立一個圖形物件
fig.add_trace(go.Scatter(x=xf, y=yf_normalized))  #新增一條線條在此圖形，將xf, yf_normalized在x, y軸上畫出
fig.update_layout(                  #更新圖形的說明
    title="範例心電訊號(6分鐘)4Hz波峰間距陣列的頻域能量分布", #幫這張圖形物件命名
    font=dict(                  #設定圖形名稱的文字
        family="Courier New, monospace",  #字體類型設定：Courier New, monospace
        size=20,               #字體大小設定：20
        color="RebeccaPurple"         #字體顏色設定：RebeccaPurple
    )
)
fig.show()                     #顯示圖形

#5. 學術統計量化心率變異率數值



> 將量化好的心跳頻域資料分門別類統計出來吧！

> 將上面頻域能量分布中，各個心率變異率規定的頻率區間加總起來，看看區間內的數值是多少吧！


> 學術統計心率變異率數值


*   Total power(TP)(整體能量): 0~0.4 Hz
*   Very low frequency power(VLFP)(極低頻能量): 0~0.04 Hz
*   Low frequency power(LFP)(低頻能量): 0.04~0.15 Hz
*   High frequency power(HFP)(高頻能量): 0.15~0.4 Hz



> 標準化與比值的分析數據



*   Normalized LFP(nLF)(標準化低頻能量) : LFP/(TP-VLFP)*100
*   Normalized HFP(nHFP)(標準化高頻能量) : HFP/(TP-VLFP)*100
*   Low high ratio(LHR)(高低頻能量比) = LFP/HFP


In [ ]:
#轉換到頻域後可以透過xf對應到該段頻率的數值
#正如畫圖的對應，xf[n]對應到的就是yf_normalized[n]的數值

n_VLFP = 0            #尋找VLFP(0.04Hz)所代表的xf[n]位置
counter = 0           #宣告一個計數器累加協助尋找n_VLFP
for i in xf:          #將xf取迴圈
  if(i <= 0.04):        #只要xf還小於0.04 Hz就繼續用counter數值取代n_VLFP
    n_VLFP = counter
  counter = counter + 1         #往上累加直到xf迴圈結束

VLFP = 0                  #預先建立VLFP的變數
counter2 = 0                 #宣告一個計數器協助累加yf_normalized數值
for i in yf_normalized:           #將yf_normalized取迴圈
  if(counter2 <= n_VLFP):         #只要counter2還小於n_VLFP就繼續累加迴圈內的yf_normalized數值給VLFP
    VLFP = i + VLFP
  counter2 = counter2 + 1        #往上累加直到yf迴圈結束

In [ ]:
#根據上面VLFP的取值方式，將TP、LFP、HFP、nLFP、nHFP、LHR都計算出來吧！



